# CRUSE Speech Enhancement — Demo

Loads a trained checkpoint and runs it on one noisy test mixture so you can listen to noisy vs. enhanced vs. clean audio inline.

Run this from the repository root (or add the repo root to `sys.path`).

In [ ]:
import sys
sys.path.append('..')  # repo root, so `src` is importable

import torch
from IPython.display import Audio, display

from src.model import CRUSE
from src.data import dataloader_test
from src.audio import make_sqrt_hann_window, stft_batch, istft_batch, log_power_spec

In [ ]:
# --- Config ---
fs = 16000
n_fft = 256
win_length = int(0.016 * fs)
hop_length = win_length // 2

checkpoint_path = '../checkpoints/cruse_best.pth'
clean_test_folder = '../Signals/Clean_Test'  # point this at your own data
noise_folder = '../Signals/Noise'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# --- Load model + weights ---
# Because the bottleneck GRU is now created inside __init__ (not lazily on
# the first forward pass), the checkpoint can be loaded directly -- no dummy
# forward pass needed.
model = CRUSE(input_freq_bins=n_fft // 2 + 1).to(device)
state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state)
model.eval()

In [ ]:
# --- Grab one test mixture and enhance it ---
test_loader = dataloader_test(clean_test_folder, noise_folder, fs, batch_size=1, shuffle=False)
window = make_sqrt_hann_window(win_length, device=device)

noisy, clean = next(iter(test_loader))
noisy, clean = noisy.to(device), clean.to(device)
length = noisy.shape[-1]

with torch.no_grad():
    xn = stft_batch(noisy, n_fft, hop_length, win_length, window)
    feat = log_power_spec(xn).permute(0, 2, 1).unsqueeze(1)
    gain = model(feat).squeeze(1).permute(0, 2, 1).contiguous()
    xhat = istft_batch(xn * gain, n_fft, hop_length, win_length, window, length=length)

In [ ]:
print('Noisy')
display(Audio(noisy.squeeze().cpu().numpy(), rate=fs))

print('Enhanced')
display(Audio(xhat.squeeze().cpu().numpy(), rate=fs))

print('Clean (reference)')
display(Audio(clean.squeeze().cpu().numpy(), rate=fs))